In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(
    "Movie_revenue_dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.shape)

(42398, 12)


In [ ]:
df.head()

,Unnamed: 0,vote_average,vote_count,release_date,revenue,runtime,budget,original_language,overview,popularity,genre,spoken_languages
0,0,8.364,34495.0,2010-07-15,8.255328e+08,148.0,160000000.0,en,"Cobb, a skilled thief who commits corporate es...",83.952,"Action, Science Fiction, Adventure","English, French, Japanese, Swahili"
1,1,8.417,32571.0,2014-11-05,7.017292e+08,169.0,165000000.0,en,The adventures of a group of explorers who mak...,140.241,"Adventure, Drama, Science Fiction",English
2,2,8.512,30619.0,2008-07-16,1.004558e+09,152.0,185000000.0,en,Batman raises the stakes in his war on crime. ...,130.643,"Drama, Action, Crime, Thriller","English, Mandarin"
3,3,7.573,29815.0,2009-12-15,2.923706e+09,162.0,237000000.0,en,"In the 22nd century, a paraplegic Marine is di...",79.932,"Action, Adventure, Fantasy, Science Fiction","English, Spanish"
4,4,7.710,29166.0,2012-04-25,1.518816e+09,143.0,220000000.0,en,When an unexpected enemy emerges and threatens...,98.082,"Science Fiction, Action, Adventure","English, Hindi, Russian"


In [ ]:
df.shape

(42398, 12)

In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
vote_average,0
vote_count,0
release_date,10
revenue,0
runtime,0
budget,0
original_language,0
overview,329
popularity,0


In [ ]:
df = df.dropna(subset=["revenue"])

In [ ]:
df['revenue'].duplicated().sum()

np.int64(31573)

In [ ]:
df.head(2)

,Unnamed: 0,vote_average,vote_count,release_date,revenue,runtime,budget,original_language,overview,popularity,genre,spoken_languages
0,0,8.364,34495.0,2010-07-15,825532764.0,148.0,160000000.0,en,"Cobb, a skilled thief who commits corporate es...",83.952,"Action, Science Fiction, Adventure","English, French, Japanese, Swahili"
1,1,8.417,32571.0,2014-11-05,701729206.0,169.0,165000000.0,en,The adventures of a group of explorers who mak...,140.241,"Adventure, Drama, Science Fiction",English


In [ ]:
df = df.drop(columns=["Unnamed: 0","spoken_languages","overview","genre","original_language"])

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df["year"] = df["release_date"].dt.year
df["month"] = df["release_date"].dt.month

In [ ]:
df["year"] = df["year"].fillna(df["year"].median())
df["month"] = df["month"].fillna(df["month"].mode()[0])

In [ ]:
df = df.drop(columns=["release_date"])

In [ ]:
num_cols = [
    "vote_average",
    "vote_count",
    "revenue",
    "runtime",
    "budget",
    "popularity"
]

for col in num_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}")
    print(f"Outliers : {len(outliers)}")
    print("-"*40)

vote_average
Outliers : 679
----------------------------------------
vote_count
Outliers : 6191
----------------------------------------
revenue
Outliers : 9994
----------------------------------------
runtime
Outliers : 4724
----------------------------------------
budget
Outliers : 9572
----------------------------------------
popularity
Outliers : 2980
----------------------------------------


In [ ]:
import numpy as np

df["vote_count"] = np.log1p(df["vote_count"])

df["revenue"] = np.log1p(df["revenue"])

df["budget"] = np.log1p(df["budget"])

df["popularity"] = np.log1p(df["popularity"])

In [ ]:
df = df.dropna(subset=["revenue"]).reset_index(drop=True)

In [ ]:
df["budget_per_min"] = df["budget"] / (df["runtime"] + 1)

df["popularity_score"] = df["popularity"] * df["vote_average"]

df["rating_votes"] = df["vote_average"] * df["vote_count"]

df["budget_popularity"] = df["budget"] * df["popularity"]

df["quarter"] = ((df["month"] - 1) // 3) + 1


In [ ]:
Q1 = df["runtime"].quantile(0.25)
Q3 = df["runtime"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    (df["runtime"] >= lower) &
    (df["runtime"] <= upper)
]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler


In [ ]:
df.columns

Index(['vote_average', 'vote_count', 'revenue', 'runtime', 'budget',
       'popularity', 'year', 'month', 'budget_per_min', 'popularity_score',
       'rating_votes', 'budget_popularity', 'quarter'],
      dtype='object')

In [ ]:
df = df.dropna(subset=["revenue"])

In [ ]:
df = df.dropna(subset=["revenue"]).reset_index(drop=True)

In [ ]:
print(df[df["revenue"].isna()])

Empty DataFrame
Columns: [vote_average, vote_count, revenue, runtime, budget, popularity, year, month, budget_per_min, popularity_score, rating_votes, budget_popularity, quarter]
Index: []


In [ ]:
X = df.drop("revenue",axis=1)
y = df['revenue']

In [ ]:
num_cols = X.select_dtypes(exclude=["object"]).columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(30139, 12)
(7535, 12)


In [ ]:
print(X.columns)

Index(['vote_average', 'vote_count', 'runtime', 'budget', 'popularity', 'year',
       'month', 'budget_per_min', 'popularity_score', 'rating_votes',
       'budget_popularity', 'quarter'],
      dtype='object')


In [ ]:
print(np.isnan(X_train).sum())
print(np.isnan(y_train).sum())

0
0


In [ ]:
print(df["revenue"].isna().sum())
print(df["revenue"].isnull().sum())

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

In [ ]:
model.add(Dense(256, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dense(128, activation="relu"))
model.add(Dense(64, activation="relu"))
model.add(Dense(32, activation="relu"))
model.add(Dense(1))

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

In [ ]:
model.metrics_names

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=512,
    callbacks=[early_stop]
)

In [ ]:
loss, mae = model.evaluate(X_test, y_test)

print("Loss:", loss)
print("MAE:", mae)

In [ ]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)

print("R2 Score:", round(r2_score(y_test, y_pred), 2))

In [ ]:
import joblib


model.save("/content/drive/MyDrive/movie_revenue_model.keras")

joblib.dump(scaler, "/content/drive/MyDrive/movie_revenue_scaler.pkl")

print("Model Saved Successfully!")

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.isnull().sum().sum())
print(np.isnan(X_train).sum())
print(np.isnan(y_train).sum())
print(np.isinf(X_train).sum(), np.isinf(y_train).sum())